# **Langgraph Agents Evaluation - Websearch Tools, Token usages, Information Delivery**

### Rafael Galán López

## **Introduccion y Marco de Evaluacion**

El presente notebook (`notebooks/agent_evaluations/langgraph_agents_evaluation.ipynb`) tiene como objetivo realizar un analisis tecnico y comparativo sobre el rendimiento de los agentes basados en LangGraph desarrollados en el modulo `src/rafael_langgraph_websearchagent/agents/`.

---

### 1. Foco de Evaluacion Principal: `A002_langgraph_agent_refined.py`

En la primera parte de este estudio nos centramos en evaluar a fondo el agente Refined (`A002`), el cual representa la base de los modelos Refined Plus `A003` y Refined Plus v2 `A004`. Analizaremos el comportamiento de su herramienta de busqueda web (`TavilySearch`) bajo los siguientes ejes clave:

1. **Estructura y Payload de la `search_tool`:** Analisis de la informacion devuelta por Tavily, la API inyectada en el prompt inicial mediante `bind_tools()` y el coste en tokens del `ToolMessage`.
2. **Generacion de Snippets y Score:** Evaluacion del metodo de extraccion de texto, la precision del score de similitud semantica y la relevancia real de los fragmentos recuperados frente a la consulta del usuario.
3. **Fidelidad Informativa vs. Alucinacion:** Diagnostico sobre como el LLM procesa las observaciones devueltas, identificando riesgos de sintesis incompleta o relleno con conocimiento estatico.

---

### 2. Evolucion y Hoja de Ruta de Optimizacion

Tras diagnosticar las limitaciones de la version Refined (`A002`), el notebook aborda la transicion hacia las versiones mejoradas:

* Agent Refined Plus v1 (`A003_langgraph_agent_refined_plus.py`): Evaluacion de la desinfeccion de payload (`sanitize_and_filter_payload`), truncado de respuesta y el filtro semantico estricto con Pydantic (`relevance_grader`).
* Agent Refined Plus v2 (`A004_langgraph_agent_refined_plusv2.py`): Analisis de la version final optimizada, demostrando la flexibilizacion del evaluador de relevancia y la aplicacion de un limite de iteraciones (`MAX_LLM_CALLS = 4`) para maximizar la precision reduciendo drasticamente el consumo de tokens.

## **Configuracion del Entorno e Importacion de Modulos**

En esta celda inicial se cargan las dependencias del proyecto, las variables de entorno necesarias para autenticar las peticiones a los modelos de lenguaje (LLM) y al motor de busqueda web, asi como las herramientas y agentes desarrollados en el paquete `src.rafael_langgraph_websearchagent`.

#### Requisitos de API Keys en `.env`
Para la correcta ejecucion de las celdas de evaluacion, es necesario contar con un archivo `.env` en la raiz del proyecto que contenga las siguientes claves de API[cite: 7]:

* `ANTHROPIC_API_KEY`: Clave de acceso al modelo de lenguaje (Claude de Anthropic).
* `TAVILY_API_KEY`: Clave de acceso al motor de busqueda y extraccion web (Tavily).

In [1]:
import sys
from dotenv import load_dotenv

# Añadir la raíz del proyecto al sys.path para importar correctamente el paquete local
sys.path.append("../../")

# Cargar API Keys de Anthropic y Tavily
load_dotenv()

# Importaciones de LangChain y LangGraph
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch, TavilyExtract
from langchain.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

# Importación de los agentes locales desde el paquete
from src.rafael_langgraph_websearchagent.agents.A002_langgraph_agent_refined import agent as agent_refined
from src.rafael_langgraph_websearchagent.agents.A003_langgraph_agent_refined_plus import agent as agent_plus_v1
from src.rafael_langgraph_websearchagent.agents.A004_langgraph_agent_refined_plusv2 import agent as agent_plus_v2

print("✅ Entorno e importaciones cargadas correctamente.")

## **1. Evaluacion del Agente A002_langgraph_agent_refined y de la search_tool**
En esta seccion se analiza la arquitectura y el comportamiento de la herramienta de busqueda web (`TavilySearch`) en el agente Refined (`A002_langgraph_agent_refined.py`), evaluando la estructura de los datos devueltos, la especificacion de la API expuesta al modelo y el impacto computacional en el estado.

### **1.1 Flujo de Ejecución de la Búsqueda Web en A002 Refined**

El proceso desde que el agente decide buscar en internet hasta que procesa la respuesta sigue tres etapas clave:

**1. Consulta y Formato de Respuesta**
El modelo formula una consulta a la API de Tavily. La herramienta devuelve una lista de diccionarios JSON (`list[dict]`) donde cada elemento contiene atributos estándar: `title` (título de la web), `url` (enlace original), `content` (snippet de texto) y `score` (puntuación de relevancia).

**2. Integración en el Estado (ToolMessage)**
LangGraph recibe la respuesta JSON, la convierte en un texto plano (string serializado) y la inyecta automáticamente en el historial de la conversación empaquetada como un objeto `ToolMessage`. Este mensaje asocia el contenido extraído (`content`) con el identificador único de la petición (`tool_call_id`).

**3. Impacto en el Consumo de Tokens**
En la versión `A002_refined` no existe ningún filtro por relevancia ni truncado de texto. Todo el payload se vuelca íntegro en el estado, lo que inyecta miles de caracteres de golpe y dispara drásticamente el consumo de tokens de entrada (`prompt_tokens`) en las siguientes iteraciones del agente.